In [ ]:
import pandas as pd

# Load the Excel file
df = pd.read_excel('NASA_Vulnerability_FinalVersion.xlsx')  # replace with your actual Excel file name

# Save it as a CSV file
df.to_csv('nasa_data_file_final.csv', index=False)  # index=False to avoid writing row numbers


In [ ]:
df = pd.read_csv('nasa_data_file.csv')
df.info()

In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize


devign_df = pd.read_csv("devign.csv")
nasa_df    = pd.read_csv("nasa_data_file.csv")


def extract_structure_features(code):
    features = {}

    features['length'] = len(code)
    features['lines'] = code.count('\n')
    features['num_func_calls'] = len(re.findall(r'\b\w+\s*\(', code))
    features['num_if'] = code.count("if")
    features['num_for'] = code.count("for")
    features['num_while'] = code.count("while")
    features['num_return'] = code.count("return")
    features['has_signature'] = bool(re.search(r'\b\w+\s+\**\w+\s*\([^)]*\)\s*{', code))
    features['uses_pointer'] = '*' in code
    features['uses_struct'] = '->' in code or '.' in code
    features['num_assignments'] = code.count("=")

    return features


devign_features = devign_df['func'].apply(extract_structure_features)
nasa_features   = nasa_df['Code'].apply(extract_structure_features)


vec = DictVectorizer(sparse=False)
devign_vec = vec.fit_transform(devign_features)
nasa_vec   = vec.transform(nasa_features)


devign_vec = normalize(devign_vec)
nasa_vec   = normalize(nasa_vec)

#cosine_similarity
similarity_scores = cosine_similarity(nasa_vec, devign_vec)
max_similarities = similarity_scores.max(axis=1)


nasa_df['devign_similarity'] = max_similarities
nasa_df['devign_like'] = nasa_df['devign_similarity'] >= 0.85  # You can adjust threshold


nasa_df[['Code', 'devign_similarity', 'devign_like']].to_csv("structural_similarity_results.csv", index=False)
nasa_df[['Code', 'devign_similarity', 'devign_like']].head()


In [ ]:
nasa_df['devign_like'].unique()

In [ ]:
sim =pd.read_csv("structural_similarity_results.csv")
sim.info()

In [ ]:
sim['devign_like'].unique()

In [ ]:
import os
import pandas as pd
from clang.cindex import Config, Index, CursorKind


Config.set_library_file(r"C:\Program Files\LLVM\bin\libclang.dll")


df = pd.read_csv("devign.csv")


from clang.cindex import Index, CursorKind

def clang_is_complete_function(code):
    with open("temp.c", "w") as f:
        f.write(code)
    index = Index.create()
    try:
        tu = index.parse("temp.c")
        for node in tu.cursor.walk_preorder():
            if node.kind == CursorKind.FUNCTION_DECL:
                return True
        return False
    except:
        return False


df["is_complete_function"] = df["func"].apply(is_complete_c_function)


df.to_csv("devign_function_check.csv", index=False)


In [ ]:
ds= pd.read_csv("devign_function_check.csv")
ds.info()

In [ ]:
ds['is_complete_function'].unique()

In [ ]:
from clang.cindex import Index, CursorKind

def clang_is_complete_function(code):
    with open("temp.c", "w") as f:
        f.write(code)
    index = Index.create()
    try:
        tu = index.parse("temp.c")
        for node in tu.cursor.walk_preorder():
            if node.kind == CursorKind.FUNCTION_DECL:
                return True
        return False
    except:
        return False


In [ ]:
import pandas as pd

# Load the Excel file
df = pd.read_excel('NASA_Vulnerability_FinalVersion.xlsx')  # replace with your actual Excel file name

# Save it as a CSV file
df.to_csv('nasa_data_file.csv', index=False)  # index=False to avoid writing row numbers


In [ ]:
data = pd.read_csv('nasa_data_file.csv')
data.info()

In [ ]:
import pandas as pd
import re

# === Step 1: Load your dataset ===
df = pd.read_csv("nasa_data_file.csv")  # Replace with your actual file name

# === Step 2: Define function completeness checker (no clang) ===
def is_complete_c_function(code):
    # Normalize whitespace and strip leading/trailing junk
    code = code.strip()

    # Check for valid function signature (e.g., int foo(), void bar(int x), etc.)
    has_signature = bool(re.search(
        r'\b(?:void|int|char|float|double|bool|long|short)\s+\**\w+\s*\([^)]*\)\s*{',
        code
    ))

    # Check that code ends with a closing brace
    has_closing_brace = code.endswith('}')

    # Check that there are balanced curly braces
    open_braces = code.count('{')
    close_braces = code.count('}')

    return has_signature and has_closing_brace and open_braces == close_braces

# === Step 3: Apply to your dataset ===
df["is_complete_function"] = df["Code"].apply(is_complete_c_function)

# === Step 4: Show results (optional) ===
print(" Completed function detection.")
display(df[["Code", "is_complete_function"]].head())

# === Step 5: Save to CSV (optional) ===
df.to_csv("your_dataset_with_function_check.csv", index=False)


In [ ]:
df['is_complete_function'].unique()

In [ ]:
df["is_complete_function"].value_counts()

In [ ]:
import pandas as pd
import re

# === Step 1: Load your dataset ===
data = pd.read_csv("devign.csv")  # Replace with your actual file name

# === Step 2: Define function completeness checker (no clang) ===
def is_complete_c_function(code):
    # Normalize whitespace and strip leading/trailing junk
    code = code.strip()

    # Check for valid function signature (e.g., int foo(), void bar(int x), etc.)
    has_signature = bool(re.search(
        r'\b(?:void|int|char|float|double|bool|long|short)\s+\**\w+\s*\([^)]*\)\s*{',
        code
    ))

    # Check that code ends with a closing brace
    has_closing_brace = code.endswith('}')

    # Check that there are balanced curly braces
    open_braces = code.count('{')
    close_braces = code.count('}')

    return has_signature and has_closing_brace and open_braces == close_braces

# === Step 3: Apply to your dataset ===
data["is_complete_function"] = data["func"].apply(is_complete_c_function)

# === Step 4: Show results (optional) ===
print(" Completed function detection.")
display(data[["func", "is_complete_function"]].head())

# === Step 5: Save to CSV (optional) ===


In [ ]:
data['is_complete_function'].unique()

In [ ]:
data["is_complete_function"].value_counts()